# 01. KIC 13F 표본 추출 (파일럿 P1)

`experiment_protocol.md` v1.0의 **2절 표본 구성 규칙**과 **1절 시점 규칙**만 구현한다.

| 표본 | 원천 | 10-K 적격성 기준일 | SIC |
|---|---|---|---|
| S1 대표 표본 | KIC 13F 2026-06-30 | 13F 접수일 이전 | 현재 SIC |
| S3 사건 표본 | KIC 13F 2022-06-30 | 2022-10-07 이전 (당일 제외) | 당시 공시 헤더의 `ASSIGNED-SIC` |

**실행 전 확인**
- SEC는 연락처가 담긴 User-Agent를 요구한다. 아래 `USER_AGENT`에 본인 영문 이름과 이메일을 넣는다.
- 처리 규칙을 바꾸고 싶다면 코드를 고치기 전에 프로토콜을 v1.1로 개정한다.
- `USE_FIXTURES = True`로 두면 네트워크 없이 모의 응답으로 전체 흐름을 점검할 수 있다 (실제 KIC 데이터 아님).

In [2]:
# ── 설정 ─────────────────────────────────────────────
USE_FIXTURES = False   # True: 모의 응답으로 스모크 테스트 (실제 데이터 아님)
USER_AGENT = "Blind Spot pilot Hyeseong Kim luckykhs@gachon.ac.kr"   # ← 반드시 수정 (ASCII만)

KIC_CIK = 1441689
SAMPLE_SIZE = 12

In [3]:
import sys
from datetime import date
from pathlib import Path

import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "tests")]
import kic13f as k

if USE_FIXTURES:
    from fixtures import FakeClient
    client = FakeClient()
    OUT_DIR = ROOT / "data" / "processed_fixture"
    print("⚠ 모의 응답 모드: 결과는 실제 KIC 보유내역이 아니다.")
else:
    client = k.SecClient(USER_AGENT, cache_dir=ROOT / "data" / "raw" / "sec_cache")
    OUT_DIR = ROOT / "data" / "processed"

SPECS = {
    "S1": k.SampleSpec("S1", "2026-06-30", cutoff=None, use_historical_sic=False, size=SAMPLE_SIZE),
    "S3": k.SampleSpec("S3", "2022-06-30", cutoff=date(2022, 10, 7), use_historical_sic=True, size=SAMPLE_SIZE),
}

## 1. CUSIP → CIK 수작업 대응표

이름 대조로 연결되지 않은 증권(상태 `failed`, `review`)은 `data/manual/cusip_cik_manual.csv`에 직접 추가한다.
- 열: `cusip, cik, note`
- `review` 후보는 자동 확정하지 않는다. EDGAR에서 회사를 확인한 뒤에만 이 파일에 넣는다.
- 이 파일의 변경 이력도 Git으로 남긴다.

In [4]:
MANUAL_PATH = ROOT / "data" / "manual" / ("cusip_cik_manual_fixture.csv" if USE_FIXTURES else "cusip_cik_manual.csv")
if MANUAL_PATH.exists():
    manual = pd.read_csv(MANUAL_PATH, dtype={"cusip": str})
else:
    manual = pd.DataFrame(columns=["cusip", "cik", "note"])
print(f"수작업 매핑 {len(manual)}건 ← {MANUAL_PATH.relative_to(ROOT)}")

tickers = k.load_ticker_table(client)
print(f"EDGAR 기업 목록 {len(tickers):,}건")

수작업 매핑 2건 ← data\manual\cusip_cik_manual.csv
EDGAR 기업 목록 10,445건


## 2. 표본 구성

In [5]:
results = {}
for name, spec in SPECS.items():
    res = k.build_sample(client, KIC_CIK, spec, tickers, manual)
    results[name] = res
    m = res.metadata
    print(f"[{name}] 13F {m['accession']} | 기준일 {m['report_date']} | 접수일 {m['filing_date']} "
          f"| value 단위 x{m['value_multiplier']} | 정정공시 미사용 {m['amendments_ignored']}")
    print(f"      매핑 상태 {m['mapping_status_counts']}")

[S1] 13F 0001193125-26-345356 | 기준일 2026-06-30 | 접수일 2026-08-12 | value 단위 x1 | 정정공시 미사용 []
      매핑 상태 {'exact': 440, 'failed': 202, 'review': 33, 'manual': 2}
[S3] 13F 0000950123-22-008583 | 기준일 2022-06-30 | 접수일 2022-08-12 | value 단위 x1000 | 정정공시 미사용 []
      매핑 상태 {'exact': 443, 'failed': 292, 'review': 25, 'manual': 2}


In [6]:
cols = ["rank", "cik", "edgar_name", "tickers", "sic", "value_usd", "weight", "latest_10k_filing_date"]
results["S1"].sample[cols].style.format({"value_usd": "{:,.0f}", "weight": "{:.2%}"})

,rank,cik,edgar_name,tickers,sic,value_usd,weight,latest_10k_filing_date
0,1,1045810,NVIDIA CORP,NVDA,3674,"3,290,676,138",17.93%,2026-02-25
1,2,320193,Apple Inc.,AAPL,3571,"2,998,806,377",16.34%,2025-10-31
2,3,1652044,Alphabet Inc.,"GOOGL,GOOG,GOOGM,GOOGN",7370,"2,645,300,777",14.42%,2026-02-05
3,4,789019,MICROSOFT CORP,MSFT,7372,"1,873,751,826",10.21%,2026-07-29
4,5,1018724,AMAZON COM INC,AMZN,5961,"1,666,166,060",9.08%,2026-02-06
5,6,1730168,Broadcom Inc.,AVGO,3674,"1,211,452,938",6.60%,2025-12-18
6,7,723125,MICRON TECHNOLOGY INC,MU,3674,"946,019,147",5.16%,2025-10-03
7,8,1326801,"Meta Platforms, Inc.",META,7370,"883,082,125",4.81%,2026-01-29
8,9,1318605,"Tesla, Inc.",TSLA,3711,"835,835,247",4.56%,2026-01-29
9,10,59478,ELI LILLY & Co,LLY,2834,"790,364,399",4.31%,2026-02-12


In [7]:
results["S3"].sample[cols].style.format({"value_usd": "{:,.0f}", "weight": "{:.2%}"})

,rank,cik,edgar_name,tickers,sic,value_usd,weight,latest_10k_filing_date
0,1,320193,Apple Inc.,AAPL,3571,"1,981,484,000",22.17%,2021-10-29
1,2,789019,MICROSOFT CORP,MSFT,7372,"1,776,643,000",19.88%,2022-07-28
2,3,1652044,Alphabet Inc.,"GOOGL,GOOG,GOOGM,GOOGN",7370,"1,202,834,000",13.46%,2022-02-02
3,4,1018724,AMAZON COM INC,AMZN,5961,"921,733,000",10.31%,2022-02-04
4,5,731766,UNITEDHEALTH GROUP INC,UNH,6324,"522,768,000",5.85%,2022-02-15
5,6,1318605,"Tesla, Inc.",TSLA,3711,"430,363,000",4.82%,2022-02-07
6,7,200406,JOHNSON & JOHNSON,JNJ,2834,"407,312,000",4.56%,2022-02-17
7,8,34088,EXXON MOBIL CORP,,2911,"356,272,000",3.99%,2022-02-23
8,9,80424,PROCTER & GAMBLE Co,PG,2840,"355,995,000",3.98%,2022-08-05
9,10,19617,JPMORGAN CHASE & CO,"JPM,JPM-PC,AMJB,JPM-PD,JPM-PJ,JPM-PK,JPM-PL,JPM-PM,VYLD",6021,"335,938,000",3.76%,2022-02-22


## 3. 제외 내역과 매핑 실패

In [8]:
for name, res in results.items():
    print(f"[{name}] 제외 {len(res.excluded)}건")
    display(res.excluded.sort_values("value_usd", ascending=False).head(15))

[S1] 제외 1건


,cusip,nameOfIssuer,value_usd,reason,cik
0,464287200,ISHARES TR,3.246025e+09,2026-08-12 이전 10-K 없음 (규칙 2.1-4),1100663.0


[S3] 제외 0건


,cusip,nameOfIssuer,value_usd,reason


In [9]:
for name, res in results.items():
    print(f"[{name}] 미확정 매핑 상위 (경계값 {res.p1['boundary_value_usd']:,.0f})")
    display(res.mapping_failures.head(15))

[S1] 미확정 매핑 상위 (경계값 534,768,841)


,cusip,nameOfIssuer,value_usd,cik,status,candidate
17,038222105,APPLIED MATLS INC,419961057.0,NaN,failed,NaN
24,17275R102,CISCO SYS INC,322162123.0,NaN,failed,NaN
25,949746101,WELLS FARGO & CO,297253436.0,NaN,failed,NaN
29,369604301,GE AEROSPACE,267854907.0,NaN,failed,NaN
34,060505104,BANK OF AMER CORP,223084791.0,NaN,failed,NaN
39,718172109,PHILIP MORRIS INTL INC,206561048.0,NaN,failed,NaN
45,882508104,TEXAS INSTRS INC,190914133.0,NaN,failed,NaN
53,459200101,INTERNATIONAL BUSINESS MACHS,177630515.0,NaN,review,INTERNATIONAL BUSINESS MACHINES -> CIK [51143]
54,780087102,ROYAL BK CDA,175489299.0,NaN,failed,NaN
57,46429B671,ISHARES TR,166746639.0,NaN,failed,NaN


[S3] 미확정 매핑 상위 (경계값 313,720,000)


,cusip,nameOfIssuer,value_usd,cik,status,candidate
19,532457108,LILLY ELI & CO,258149000.0,NaN,failed,NaN
23,060505104,BK OF AMERICA CORP,254205000.0,NaN,failed,NaN
28,22160K105,COSTCO WHSL CORP NEW,187137000.0,NaN,failed,NaN
30,00724F101,ADOBE SYSTEMS INCORPORATED,179163000.0,NaN,failed,NaN
32,002824100,ABBOTT LABS,174736000.0,NaN,failed,NaN
33,G1151C101,ACCENTURE PLC IRELAND,165967000.0,NaN,failed,NaN
34,609207105,MONDELEZ INTL INC,165140000.0,NaN,failed,NaN
36,17275R102,CISCO SYS INC,164246000.0,NaN,failed,NaN
39,548661107,LOWES COS INC,159659000.0,NaN,failed,NaN
40,949746101,WELLS FARGO CO NEW,158169000.0,NaN,failed,NaN


## 4. P1 판정

성공 기준 (프로토콜 7.1): S1·S3 모두 상위 12개 선정 완료, 그리고 **12위 경계 이상의 미확정 매핑 0건**.

실패하면 결과를 보고 표본 규칙을 바꾸지 않는다. 미확정 매핑을 수작업 대응표로 해소한 뒤 재실행한다.

In [10]:
p1 = pd.DataFrame([r.p1 for r in results.values()])
display(p1)
print("P1 전체:", "통과" if p1["pass"].all() else "미통과 → 수작업 대응표 보완 후 재실행")

,sample,selected_issuers,target,boundary_value_usd,unmapped_at_or_above_boundary,pass
0,S1,12,12,534768841.0,0,True
1,S3,12,12,313720000.0,0,True


P1 전체: 통과


In [11]:
for res in results.values():
    path = k.save_result(res, OUT_DIR)
    print("저장:", path.relative_to(ROOT))

저장: data\processed\S1
저장: data\processed\S3


## 5. 해석 시 유의점

- 이 표본은 **KIC가 SEC에 공시한 13F 보유내역에서 추출해 비중을 재정규화한 샘플**이며, KIC 전체 포트폴리오를 대표하지 않는다.
- S3 비중은 2022-06-30 기준값이며, 이후 사건일(2022-10-07)까지의 매매는 반영되지 않는다.
- 13F 범위상 미국 상장주식 롱포지션만 포함되고, 10-K 미제출 발행사(ETF, 20-F 제출 해외 발행사)는 규칙에 따라 제외된다.
- 다음 단계(P2): S1 상위 5개 발행사에 대해 사람이 먼저 정답 라벨을 붙인 뒤 공시 노출 추출을 실행한다.

In [12]:
from datetime import datetime

empty_manual = pd.DataFrame(columns=["cusip", "cik", "note"])
for name, spec in SPECS.items():
    r0 = k.build_sample(client, KIC_CIK, spec, tickers, empty_manual)
    m, p = r0.metadata, r0.p1
    status = "통과" if p["pass"] else f"미통과 (경계 이상 미확정 매핑 {p['unmapped_at_or_above_boundary']}건)"
    print(f"| {datetime.now():%Y-%m-%d %H:%M} (재현) | P1-{name} | 13F {m['accession']} (기준일 {m['report_date']}) "
          f"| {status}, 선정 {p['selected_issuers']}/{p['target']} | 수작업 매핑 0건 |")
    # 미통과 원인이 된 증권 목록 (기록용)
    boundary = p["boundary_value_usd"]
    display(r0.mapping_failures[r0.mapping_failures["value_usd"] >= boundary])

| 2026-10-13 21:37 (재현) | P1-S1 | 13F 0001193125-26-345356 (기준일 2026-06-30) | 미통과 (경계 이상 미확정 매핑 1건), 선정 12/12 | 수작업 매핑 0건 |


,cusip,nameOfIssuer,value_usd,cik,status,candidate
1,464287200,ISHARES TR,3.246025e+09,NaN,failed,NaN


| 2026-10-13 21:37 (재현) | P1-S3 | 13F 0000950123-22-008583 (기준일 2022-06-30) | 미통과 (경계 이상 미확정 매핑 1건), 선정 12/12 | 수작업 매핑 0건 |


,cusip,nameOfIssuer,value_usd,cik,status,candidate
8,30231G102,EXXON MOBIL CORP,356272000.0,NaN,failed,NaN
